#Install Libraries

In [1]:
!pip install transformers torch scikit-learn -q

#Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Imports Library

In [3]:
import json
import re
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded!")

Libraries loaded!


#Config

In [4]:
class Config:
    LAPTOP_FILE     = "/content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/laptop_train.jsonl"
    RESTAURANT_FILE = "/content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/restaurant_train.jsonl"
    MODEL_SAVE_PATH = "/content/drive/MyDrive/DATASET/NLP/absa_best_model.pt"

    # If inside a folder (e.g. folder = "ABSA_Data"), change to:
    # LAPTOP_FILE     = "/content/drive/MyDrive/ABSA_Data/laptop_train.jsonl"
    # RESTAURANT_FILE = "/content/drive/MyDrive/ABSA_Data/restaurant_train.jsonl"
    # MODEL_SAVE_PATH = "/content/drive/MyDrive/ABSA_Data/absa_best_model.pt"

    # ── Model ──────────────────────────────────────────────────
    BERT_MODEL   = "bert-base-uncased"
    MAX_SEQ_LEN  = 128
    HIDDEN_SIZE  = 768

    # ── Training ───────────────────────────────────────────────
    BATCH_SIZE   = 16
    EPOCHS       = 8
    LR           = 1e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.05
    SEED         = 42

    # ── Validation Split ───────────────────────────────────────
    # 15% used ONLY to track best epoch and save best model
    VAL_SPLIT    = 0.15

    # ── Loss Weights ───────────────────────────────────────────
    ALPHA_ASP    = 1.0
    ALPHA_OPN    = 1.0
    ALPHA_VA     = 1.0

    # ── VA Scale (dataset: 1 to 9) ─────────────────────────────
    VA_MIN       = 1.0
    VA_MAX       = 9.0

    # ── BIO Tag Maps ───────────────────────────────────────────
    ASP_TAGS = {"O": 0, "B-ASP": 1, "I-ASP": 2}
    OPN_TAGS = {"O": 0, "B-OPN": 1, "I-OPN": 2}
    IDX2ASP  = {v: k for k, v in ASP_TAGS.items()}
    IDX2OPN  = {v: k for k, v in OPN_TAGS.items()}

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

print(f"Device        : {cfg.DEVICE}")
print(f"Laptop file   : {cfg.LAPTOP_FILE}")
print(f"Restaurant    : {cfg.RESTAURANT_FILE}")
print(f"Model save to : {cfg.MODEL_SAVE_PATH}")

Device        : cuda
Laptop file   : /content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/laptop_train.jsonl
Restaurant    : /content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/restaurant_train.jsonl
Model save to : /content/drive/MyDrive/DATASET/NLP/absa_best_model.pt


#Data Loading & Preprocessing

In [5]:
def clean_text(text: str) -> str:
    text = re.sub(r"` ` (.*?) ` `", r"\1", text)
    text = re.sub(r"`", "", text)
    return text.strip()


def load_jsonl(path: str):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_va(val: float) -> float:
    return (val - cfg.VA_MIN) / (cfg.VA_MAX - cfg.VA_MIN)


def denormalize_va(val: float) -> float:
    return val * (cfg.VA_MAX - cfg.VA_MIN) + cfg.VA_MIN


def find_span_in_tokens(text_tokens, span_words):
    span_lower = [w.lower() for w in span_words]
    text_lower = [w.lower() for w in text_tokens]
    n = len(span_lower)
    for i in range(len(text_lower) - n + 1):
        if text_lower[i:i+n] == span_lower:
            return i, i + n - 1
    return None


def build_bio_labels(text_tokens, span_str, tag_b, tag_i, tag_o):
    labels = [tag_o] * len(text_tokens)
    if span_str == "NULL" or span_str is None:
        return labels
    result = find_span_in_tokens(text_tokens, span_str.lower().split())
    if result is None:
        return labels
    start, end = result
    labels[start] = tag_b
    for i in range(start + 1, end + 1):
        labels[i] = tag_i
    return labels


def prepare_records(raw_records):
    examples = []
    for rec in raw_records:
        text        = clean_text(rec["Text"])
        word_tokens = text.split()
        for quad in rec["Quadruplet"]:
            v, a = [float(x) for x in quad["VA"].split("#")]
            examples.append({
                "text"       : text,
                "word_tokens": word_tokens,
                "asp_labels" : build_bio_labels(
                    word_tokens, quad["Aspect"],
                    cfg.ASP_TAGS["B-ASP"], cfg.ASP_TAGS["I-ASP"], cfg.ASP_TAGS["O"]
                ),
                "opn_labels" : build_bio_labels(
                    word_tokens, quad["Opinion"],
                    cfg.OPN_TAGS["B-OPN"], cfg.OPN_TAGS["I-OPN"], cfg.OPN_TAGS["O"]
                ),
                "valence"    : normalize_va(v),
                "arousal"    : normalize_va(a),
                "asp_str"    : quad["Aspect"],
                "opn_str"    : quad["Opinion"],
            })
    return examples

#PyTorch Dataset

In [6]:
class ABSADataset(Dataset):
    def __init__(self, examples, tokenizer, max_len):
        self.examples  = examples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex    = self.examples[idx]
        words = ex["word_tokens"]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        word_ids     = encoding.word_ids(batch_index=0)
        asp_subword  = []
        opn_subword  = []
        prev_word_id = None

        for wid in word_ids:
            if wid is None:
                asp_subword.append(-100)
                opn_subword.append(-100)
            elif wid != prev_word_id:
                asp_subword.append(ex["asp_labels"][wid] if wid < len(ex["asp_labels"]) else 0)
                opn_subword.append(ex["opn_labels"][wid] if wid < len(ex["opn_labels"]) else 0)
            else:
                asp_subword.append(-100)
                opn_subword.append(-100)
            prev_word_id = wid

        asp_subword = (asp_subword + [-100] * self.max_len)[:self.max_len]
        opn_subword = (opn_subword + [-100] * self.max_len)[:self.max_len]

        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "token_type_ids" : encoding["token_type_ids"].squeeze(0),
            "asp_labels"     : torch.tensor(asp_subword, dtype=torch.long),
            "opn_labels"     : torch.tensor(opn_subword, dtype=torch.long),
            "valence"        : torch.tensor(ex["valence"], dtype=torch.float),
            "arousal"        : torch.tensor(ex["arousal"], dtype=torch.float),
        }

#Model Architecture

In [7]:
class ABSAModel(nn.Module):
    """
    BERT (shared encoder)
      |
      +-- Aspect Head  : Linear(768->3)   -> BIO tags O/B-ASP/I-ASP
      +-- Opinion Head : Linear(768->3)   -> BIO tags O/B-OPN/I-OPN
      |
      +-- Span Extraction (mean pool over B/I tokens)
      |
      +-- VA MLP : [asp_vec || opn_vec] 1536 -> 768 -> 256 -> 2
                   -> (Valence, Arousal)
    """

    def __init__(self, bert_model_name, hidden_size=768, num_bio_tags=3, dropout=0.2):
        super().__init__()
        self.bert         = BertModel.from_pretrained(bert_model_name)
        self.dropout      = nn.Dropout(dropout)
        self.aspect_head  = nn.Linear(hidden_size, num_bio_tags)
        self.opinion_head = nn.Linear(hidden_size, num_bio_tags)
        self.va_head      = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
            nn.Sigmoid()
        )
        asp_w = torch.tensor([1.0, 8.0, 6.0])  # O=1, B-ASP=8, I-ASP=6
        opn_w = torch.tensor([1.0, 8.0, 6.0])  # O=1, B-OPN=8, I-OPN=6
        self.asp_ce_loss = nn.CrossEntropyLoss(ignore_index=-100, weight=asp_w)
        self.opn_ce_loss = nn.CrossEntropyLoss(ignore_index=-100, weight=opn_w)
        self.mse_loss    = nn.MSELoss()

    def forward(self, input_ids, attention_mask, token_type_ids,
                asp_labels=None, opn_labels=None, valence=None, arousal=None):

        bert_out   = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        seq_out    = self.dropout(bert_out.last_hidden_state)
        asp_logits = self.aspect_head(seq_out)
        opn_logits = self.opinion_head(seq_out)
        asp_vec    = self._span_vector(seq_out, asp_logits, attention_mask)
        opn_vec    = self._span_vector(seq_out, opn_logits, attention_mask)
        va_pred    = self.va_head(torch.cat([asp_vec, opn_vec], dim=-1))

        loss = None
        if asp_labels is not None:
            self.asp_ce_loss.weight = self.asp_ce_loss.weight.to(input_ids.device)
            self.opn_ce_loss.weight = self.opn_ce_loss.weight.to(input_ids.device)

            B, L, C = asp_logits.shape
            loss = (cfg.ALPHA_ASP * self.asp_ce_loss(asp_logits.view(B*L,C), asp_labels.view(B*L))
                  + cfg.ALPHA_OPN * self.opn_ce_loss(opn_logits.view(B*L,C), opn_labels.view(B*L))
                  + cfg.ALPHA_VA  * self.mse_loss(va_pred,
                        torch.stack([valence, arousal], dim=-1)))

        return {"loss": loss, "asp_logits": asp_logits,
                "opn_logits": opn_logits, "va_pred": va_pred}

    def _span_vector(self, seq_out, logits, attention_mask):
        preds    = torch.argmax(logits, dim=-1)
        mask     = ((preds==1)|(preds==2)).float() * attention_mask.float()
        sum_vec  = (seq_out * mask.unsqueeze(-1)).sum(dim=1)
        mean_vec = sum_vec / mask.sum(dim=1, keepdim=True).clamp(min=1e-9)
        no_span  = (mask.sum(dim=1)==0).unsqueeze(-1).float()
        return mean_vec * (1-no_span) + seq_out[:,0,:] * no_span

#Span Extraction & Pairing Module

In [8]:
def extract_spans_from_bio(tokens, bio_tags):
    spans, i = [], 0
    while i < len(tokens):
        if bio_tags[i] in ("B-ASP", "B-OPN"):
            start    = i
            cont_tag = "I-ASP" if bio_tags[i]=="B-ASP" else "I-OPN"
            i += 1
            while i < len(tokens) and bio_tags[i] == cont_tag:
                i += 1
            spans.append((start, i-1, " ".join(tokens[start:i])))
        else:
            i += 1
    return spans


def clause_split(tokens):
    split_words = {"but", ",", "and", "however", "although", "though", "while", "yet"}
    clauses, start = [], 0
    for i, tok in enumerate(tokens):
        if tok.lower() in split_words and i > start:
            clauses.append((start, i-1))
            start = i+1
    if start < len(tokens):
        clauses.append((start, len(tokens)-1))
    return clauses if clauses else [(0, len(tokens)-1)]


def pair_aspects_opinions(tokens, asp_spans, opn_spans):
    if not asp_spans and not opn_spans:
        return [{"Aspect": "NULL", "Opinion": "NULL"}]
    if not asp_spans:
        return [{"Aspect": "NULL", "Opinion": o[2]} for o in opn_spans]
    if not opn_spans:
        return [{"Aspect": a[2], "Opinion": "NULL"} for a in asp_spans]

    clauses, pairs, used_opn = clause_split(tokens), [], set()

    for asp in asp_spans:
        asp_mid    = (asp[0]+asp[1])/2
        asp_clause = next((c for c in clauses if c[0]<=asp[0]<=c[1]), None)
        matched    = []
        if asp_clause:
            for j, opn in enumerate(opn_spans):
                if asp_clause[0] <= opn[0] <= asp_clause[1]:
                    matched.append((j, opn))
        if matched:
            for j, opn in matched:
                pairs.append({"Aspect": asp[2], "Opinion": opn[2]})
                used_opn.add(j)
        else:
            nearest = min(enumerate(opn_spans),
                          key=lambda x: abs((x[1][0]+x[1][1])/2 - asp_mid))
            pairs.append({"Aspect": asp[2], "Opinion": nearest[1][2]})
            used_opn.add(nearest[0])

    for j, opn in enumerate(opn_spans):
        if j not in used_opn:
            pairs.append({"Aspect": "NULL", "Opinion": opn[2]})

    return pairs if pairs else [{"Aspect": "NULL", "Opinion": "NULL"}]

#Train & Eval Functions

In [9]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        out = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["token_type_ids"].to(device),
            batch["asp_labels"].to(device),
            batch["opn_labels"].to(device),
            batch["valence"].to(device),
            batch["arousal"].to(device),
        )
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out["loss"].item()
    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    all_asp_p, all_asp_t = [], []
    all_opn_p, all_opn_t = [], []
    all_va_p,  all_va_t  = [], []

    with torch.no_grad():
        for batch in loader:
            out = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["token_type_ids"].to(device),
                batch["asp_labels"].to(device),
                batch["opn_labels"].to(device),
                batch["valence"].to(device),
                batch["arousal"].to(device),
            )
            total_loss += out["loss"].item()

            asp_p = torch.argmax(out["asp_logits"], dim=-1).cpu().numpy()
            opn_p = torch.argmax(out["opn_logits"], dim=-1).cpu().numpy()
            asp_t = batch["asp_labels"].numpy()
            opn_t = batch["opn_labels"].numpy()

            mask = asp_t != -100
            all_asp_p.extend(asp_p[mask].tolist())
            all_asp_t.extend(asp_t[mask].tolist())

            mask = opn_t != -100
            all_opn_p.extend(opn_p[mask].tolist())
            all_opn_t.extend(opn_t[mask].tolist())

            va_p = out["va_pred"].cpu().numpy()
            va_t = torch.stack([batch["valence"], batch["arousal"]], dim=-1).numpy()
            all_va_p.extend(va_p.tolist())
            all_va_t.extend(va_t.tolist())

    asp_f1 = f1_score(all_asp_t, all_asp_p, average="macro", zero_division=0)
    opn_f1 = f1_score(all_opn_t, all_opn_p, average="macro", zero_division=0)
    va_mae = np.mean(np.abs(np.array(all_va_p) - np.array(all_va_t)))

    return {"loss": total_loss/len(loader), "asp_f1": asp_f1,
            "opn_f1": opn_f1, "va_mae": va_mae}

#Main Training

In [10]:
def main():
    print("=" * 65)
    print("   ABSA MODEL — TRAINING START")
    print("=" * 65)

    # 1. Load from Drive
    print("\n[1/6] Loading data from Google Drive...")
    laptop_raw     = load_jsonl(cfg.LAPTOP_FILE)
    restaurant_raw = load_jsonl(cfg.RESTAURANT_FILE)
    all_raw        = laptop_raw + restaurant_raw
    print(f"  Laptop     : {len(laptop_raw):,} sentences")
    print(f"  Restaurant : {len(restaurant_raw):,} sentences")
    print(f"  Total      : {len(all_raw):,} sentences")

    # 2. Prepare examples
    print("\n[2/6] Building examples from quadruplets...")
    all_examples = prepare_records(all_raw)
    print(f"  Total examples : {len(all_examples):,}")

    np.random.shuffle(all_examples)
    split    = int(len(all_examples) * (1 - cfg.VAL_SPLIT))
    train_ex = all_examples[:split]
    val_ex   = all_examples[split:]
    print(f"  Train      : {len(train_ex):,}  (85% - model learns from this)")
    print(f"  Validation : {len(val_ex):,}  (15% - only to save best epoch)")
    print(f"  Note       : test data is completely separate")

    # 3. Tokenizer & DataLoaders
    print(f"\n[3/6] Loading tokenizer...")
    tokenizer    = BertTokenizerFast.from_pretrained(cfg.BERT_MODEL)
    train_loader = DataLoader(ABSADataset(train_ex, tokenizer, cfg.MAX_SEQ_LEN),
                              batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader   = DataLoader(ABSADataset(val_ex, tokenizer, cfg.MAX_SEQ_LEN),
                              batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

    # 4. Model
    print(f"\n[4/6] Building model...")
    model        = ABSAModel(cfg.BERT_MODEL, cfg.HIDDEN_SIZE).to(cfg.DEVICE)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params : {total_params:,}")

    # 5. Optimizer
    optimizer    = AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    total_steps  = len(train_loader) * cfg.EPOCHS
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * cfg.WARMUP_RATIO), total_steps)
    print(f"\n[5/6] Optimizer ready")

    # 6. Training loop
    print(f"\n[6/6] Training {cfg.EPOCHS} epochs...\n")
    print("=" * 80)
    print(f"  {'Ep':>3}/{cfg.EPOCHS:<3} | {'TrLoss':>7} | {'ValLoss':>7} | "
          f"{'AspF1':>6} | {'OpnF1':>6} | {'VMAE':>6} | {'Time':>4} | ETA")
    print("  " + "-" * 78)

    best_val_loss = float("inf")
    best_asp_f1   = 0.0
    best_opn_f1   = 0.0
    best_va_mae   = float("inf")
    best_epoch    = 0
    history       = []
    train_start   = time.time()

    for epoch in range(1, cfg.EPOCHS + 1):
        ep_start   = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, cfg.DEVICE)
        val_m      = eval_epoch(model, val_loader, cfg.DEVICE)
        ep_secs    = time.time() - ep_start
        elapsed    = time.time() - train_start
        eta_secs   = (elapsed / epoch) * (cfg.EPOCHS - epoch)
        eta_str    = f"{int(eta_secs//60)}m{int(eta_secs%60):02d}s"
        saved      = ""

        if val_m["loss"] < best_val_loss:
            best_val_loss = val_m["loss"]
            best_asp_f1   = val_m["asp_f1"]
            best_opn_f1   = val_m["opn_f1"]
            best_va_mae   = val_m["va_mae"]
            best_epoch    = epoch
            saved         = "  << BEST SAVED"
            torch.save({
                "epoch"      : epoch,
                "model_state": model.state_dict(),
                "val_loss"   : best_val_loss,
                "asp_f1"     : best_asp_f1,
                "opn_f1"     : best_opn_f1,
                "va_mae"     : best_va_mae,
                "bert_model" : cfg.BERT_MODEL,
            }, cfg.MODEL_SAVE_PATH)

        print(f"  {epoch:>3}/{cfg.EPOCHS:<3} | {train_loss:>7.4f} | {val_m['loss']:>7.4f} | "
              f"{val_m['asp_f1']:>6.4f} | {val_m['opn_f1']:>6.4f} | {val_m['va_mae']:>6.4f} | "
              f"{ep_secs:>3.0f}s | {eta_str}{saved}")

        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_m["loss"], "asp_f1": val_m["asp_f1"],
                        "opn_f1": val_m["opn_f1"], "va_mae": val_m["va_mae"]})

    total_t = time.time() - train_start

    # Final report
    print("\n")
    print("=" * 62)
    print("     TRAINING COMPLETE — FINAL ACCURACY REPORT")
    print("=" * 62)
    print(f"  Total time          : {int(total_t//60)}m {int(total_t%60)}s")
    print(f"  Total epochs        : {cfg.EPOCHS}")
    print(f"  Best epoch          : {best_epoch}")
    print("-" * 62)
    print(f"  BEST MODEL SCORES  (validation set)")
    print("-" * 62)
    print(f"  Val Loss            : {best_val_loss:.4f}")
    print(f"  Aspect  F1 score    : {best_asp_f1:.4f}   ({best_asp_f1*100:.1f}%)")
    print(f"  Opinion F1 score    : {best_opn_f1:.4f}   ({best_opn_f1*100:.1f}%)")
    print(f"  VA MAE  (0-1 scale) : {best_va_mae:.4f}")
    print(f"  VA MAE  (1-9 scale) : {best_va_mae * 8:.4f}")
    print("-" * 62)
    print(f"  SCORE GUIDE:")
    print(f"  Aspect/Opinion F1 > 0.70  → Good  |  > 0.80 → Excellent")
    print(f"  VA MAE (1-9 scale) < 1.0  → Good  |  < 0.5  → Excellent")
    print(f"  Best model saved to : {cfg.MODEL_SAVE_PATH}")
    print("=" * 62)

    # Full history
    print("\n  FULL EPOCH HISTORY:")
    print(f"  {'Ep':>3} | {'TrLoss':>7} | {'ValLoss':>7} | {'AspF1':>6} | {'OpnF1':>6} | {'VMAE':>6}")
    print("  " + "-" * 54)
    for h in history:
        mark = "  <- best" if h["epoch"] == best_epoch else ""
        print(f"  {h['epoch']:>3} | {h['train_loss']:>7.4f} | {h['val_loss']:>7.4f} | "
              f"{h['asp_f1']:>6.4f} | {h['opn_f1']:>6.4f} | {h['va_mae']:>6.4f}{mark}")

    print(f"\n  Model saved → {cfg.MODEL_SAVE_PATH}")
    print("  Scroll down to run INFERENCE cells!")
    return model, tokenizer

#RUN TRAINING

In [11]:
model, tokenizer = main()

   ABSA MODEL — TRAINING START

[1/6] Loading data from Google Drive...
  Laptop     : 4,076 sentences
  Restaurant : 2,284 sentences
  Total      : 6,360 sentences

[2/6] Building examples from quadruplets...
  Total examples : 9,432
  Train      : 8,017  (85% - model learns from this)
  Validation : 1,415  (15% - only to save best epoch)
  Note       : test data is completely separate

[3/6] Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


[4/6] Building model...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable params : 110,864,648

[5/6] Optimizer ready

[6/6] Training 8 epochs...

   Ep/8   |  TrLoss | ValLoss |  AspF1 |  OpnF1 |   VMAE | Time | ETA
  ------------------------------------------------------------------------------
    1/8   |  1.1082 |  0.5362 | 0.6983 | 0.6775 | 0.1205 | 198s | 23m07s  << BEST SAVED
    2/8   |  0.4619 |  0.4835 | 0.7173 | 0.6782 | 0.0927 | 206s | 20m17s  << BEST SAVED
    3/8   |  0.3812 |  0.4755 | 0.7109 | 0.6870 | 0.0879 | 206s | 17m02s  << BEST SAVED
    4/8   |  0.3387 |  0.4719 | 0.7211 | 0.6982 | 0.0847 | 206s | 13m40s  << BEST SAVED
    5/8   |  0.3067 |  0.4923 | 0.7370 | 0.7043 | 0.0799 | 206s | 10m16s
    6/8   |  0.2893 |  0.5137 | 0.7361 | 0.7046 | 0.0807 | 206s | 6m51s
    7/8   |  0.2740 |  0.5337 | 0.7357 | 0.7060 | 0.0788 | 206s | 3m25s
    8/8   |  0.2648 |  0.5432 | 0.7376 | 0.7056 | 0.0784 | 206s | 0m00s


     TRAINING COMPLETE — FINAL ACCURACY REPORT
  Total time          : 27m 24s
  Total epochs        : 8
  Best epoch    

#INFERENCE CELL A — Load Saved Model

In [12]:
def load_saved_model(model_path=None):
    """
    Load the best model from Drive without retraining.
    Useful when:
      - Colab session restarted
      - Want to test on new data
      - Training already done
    """
    if model_path is None:
        model_path = cfg.MODEL_SAVE_PATH

    print(f"Loading model from Drive: {model_path}")
    ckpt      = torch.load(model_path, map_location=cfg.DEVICE)
    bert_name = ckpt.get("bert_model", cfg.BERT_MODEL)
    model     = ABSAModel(bert_name, cfg.HIDDEN_SIZE).to(cfg.DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    tokenizer = BertTokenizerFast.from_pretrained(bert_name)

    print(f"  Loaded from epoch : {ckpt['epoch']}")
    print(f"  Val Loss          : {ckpt['val_loss']:.4f}")
    print(f"  Aspect  F1        : {ckpt['asp_f1']:.4f}  ({ckpt['asp_f1']*100:.1f}%)")
    print(f"  Opinion F1        : {ckpt['opn_f1']:.4f}  ({ckpt['opn_f1']*100:.1f}%)")
    print(f"  VA MAE (1-9 scale): {ckpt['va_mae']*8:.4f}")
    print("  Model ready!")
    return model, tokenizer


#INFERENCE CELL B Core Predict Function

In [13]:
def predict(model, tokenizer, text, device=None):
    """
    Raw text -> JSON triplets

    Example:
      Input  : "the food was good but service was slow"
      Output : [
        {"Aspect": "food",    "Opinion": "good", "VA": "7.33#7.50"},
        {"Aspect": "service", "Opinion": "slow", "VA": "3.10#6.20"}
      ]
    """
    if device is None:
        device = cfg.DEVICE

    model.eval()
    text  = clean_text(text)
    words = text.split()

    if not words:
        return [{"Aspect": "NULL", "Opinion": "NULL", "VA": "5.00#5.00"}]

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        max_length=cfg.MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        out = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device),
            encoding["token_type_ids"].to(device),
        )

    word_ids = encoding.word_ids(batch_index=0)
    asp_pred = torch.argmax(out["asp_logits"], dim=-1).squeeze(0).cpu().numpy()
    opn_pred = torch.argmax(out["opn_logits"], dim=-1).squeeze(0).cpu().numpy()

    word_asp, word_opn = {}, {}
    for pos, wid in enumerate(word_ids):
        if wid is not None and wid not in word_asp:
            word_asp[wid] = cfg.IDX2ASP[asp_pred[pos]]
            word_opn[wid] = cfg.IDX2OPN[opn_pred[pos]]

    asp_tags  = [word_asp.get(i, "O") for i in range(len(words))]
    opn_tags  = [word_opn.get(i, "O") for i in range(len(words))]
    asp_spans = extract_spans_from_bio(words, asp_tags)
    opn_spans = extract_spans_from_bio(words, opn_tags)
    pairs     = pair_aspects_opinions(words, asp_spans, opn_spans)

    va  = out["va_pred"].squeeze(0).cpu().numpy()
    v   = round(float(denormalize_va(va[0])), 2)
    a   = round(float(denormalize_va(va[1])), 2)

    return [{"Aspect": p["Aspect"], "Opinion": p["Opinion"], "VA": f"{v}#{a}"}
            for p in pairs]

#INFERENCE CELL C — Test Single Sentence

In [14]:
def test_single(text):
    print(f"\nInput  : {text}")
    results = predict(model, tokenizer, text)
    for r in results:
        v, a = r["VA"].split("#")
        print(f"  -> Aspect  : {r['Aspect']}")
        print(f"     Opinion : {r['Opinion']}")
        print(f"     Valence : {v}  |  Arousal : {a}")
    return results

# Change sentence here and run:
test_single("the food was very good but the service was really slow")


Input  : the food was very good but the service was really slow
  -> Aspect  : food
     Opinion : very good
     Valence : 6.01  |  Arousal : 6.74
  -> Aspect  : service
     Opinion : really slow
     Valence : 6.01  |  Arousal : 6.74


[{'Aspect': 'food', 'Opinion': 'very good', 'VA': '6.01#6.74'},
 {'Aspect': 'service', 'Opinion': 'really slow', 'VA': '6.01#6.74'}]

#INFERENCE CELL D — Test Multiple Sentence

In [15]:
def test_batch(sentences):
    print("=" * 62)
    print("  BATCH INFERENCE RESULTS")
    print("=" * 62)
    all_results = []
    for i, sent in enumerate(sentences, 1):
        print(f"\n[{i}] {sent}")
        results = predict(model, tokenizer, sent)
        for r in results:
            print(f"     Aspect={r['Aspect']}  |  Opinion={r['Opinion']}  |  VA={r['VA']}")
        all_results.append({"text": sent, "predictions": results})
    print("\n" + "=" * 62)
    return all_results

# Add/remove sentences as needed:
test_batch([
    "the food was very good but service was slow",
    "battery life is terrible but the keyboard is amazing",
    "totally worth the price",
    "get the tuna",
    "i will never come back to this place",
    "the screen is bright and performance is fast",
    "horrible experience , staff was rude",
])

  BATCH INFERENCE RESULTS

[1] the food was very good but service was slow
     Aspect=food  |  Opinion=very good  |  VA=5.02#6.46
     Aspect=service  |  Opinion=slow  |  VA=5.02#6.46

[2] battery life is terrible but the keyboard is amazing
     Aspect=battery life  |  Opinion=terrible  |  VA=6.7#7.05
     Aspect=keyboard  |  Opinion=amazing  |  VA=6.7#7.05

[3] totally worth the price
     Aspect=NULL  |  Opinion=totally worth  |  VA=7.06#7.26

[4] get the tuna
     Aspect=tuna  |  Opinion=NULL  |  VA=5.31#5.6

[5] i will never come back to this place
     Aspect=place  |  Opinion=never  |  VA=3.65#6.3

[6] the screen is bright and performance is fast
     Aspect=screen  |  Opinion=bright  |  VA=7.67#7.53
     Aspect=performance  |  Opinion=fast  |  VA=7.67#7.53

[7] horrible experience , staff was rude
     Aspect=staff  |  Opinion=rude  |  VA=2.89#6.68
     Aspect=NULL  |  Opinion=horrible  |  VA=2.89#6.68



[{'text': 'the food was very good but service was slow',
  'predictions': [{'Aspect': 'food',
    'Opinion': 'very good',
    'VA': '5.02#6.46'},
   {'Aspect': 'service', 'Opinion': 'slow', 'VA': '5.02#6.46'}]},
 {'text': 'battery life is terrible but the keyboard is amazing',
  'predictions': [{'Aspect': 'battery life',
    'Opinion': 'terrible',
    'VA': '6.7#7.05'},
   {'Aspect': 'keyboard', 'Opinion': 'amazing', 'VA': '6.7#7.05'}]},
 {'text': 'totally worth the price',
  'predictions': [{'Aspect': 'NULL',
    'Opinion': 'totally worth',
    'VA': '7.06#7.26'}]},
 {'text': 'get the tuna',
  'predictions': [{'Aspect': 'tuna', 'Opinion': 'NULL', 'VA': '5.31#5.6'}]},
 {'text': 'i will never come back to this place',
  'predictions': [{'Aspect': 'place', 'Opinion': 'never', 'VA': '3.65#6.3'}]},
 {'text': 'the screen is bright and performance is fast',
  'predictions': [{'Aspect': 'screen', 'Opinion': 'bright', 'VA': '7.67#7.53'},
   {'Aspect': 'performance', 'Opinion': 'fast', 'VA': '7

INFERENCE CELL E — Run on Full Test File

In [16]:
def run_on_test_file(test_file_path, output_path=None):
    """
    Read sir's test JSONL file, predict for each sentence,
    save results in EXACT required submission format:
      - sentence_id
      - triplets: list of (Aspect, Opinion, Valence, Arousal)

    Usage:
      run_on_test_file(
          "/content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/test_data.jsonl",
          "/content/drive/MyDrive/DATASET/NLP/submission_output.jsonl"
      )
    """
    if output_path is None:
        output_path = test_file_path.replace(".jsonl", "_submission.jsonl")

    print(f"Reading test file : {test_file_path}")
    test_records = load_jsonl(test_file_path)
    print(f"Total sentences   : {len(test_records)}")
    print("Running inference...")

    submission = []

    for i, rec in enumerate(test_records):
        text      = clean_text(rec["Text"])
        raw_preds = predict(model, tokenizer, text)

        # Build triplets in sir's required format
        triplets = []
        for p in raw_preds:
            v_str, a_str = p["VA"].split("#")
            triplets.append({
                "Aspect" : p["Aspect"],
                "Opinion": p["Opinion"],
                "Valence": float(v_str),
                "Arousal": float(a_str),
            })

        submission.append({
            "sentence_id": rec.get("ID", f"test_{i}"),
            "triplets"   : triplets,
        })

        if (i + 1) % 100 == 0:
            print(f"  Processed {i+1}/{len(test_records)}...")

    # Save as JSONL — one record per line
    with open(output_path, "w", encoding="utf-8") as f:
        for record in submission:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"\nDone!")
    print(f"Total records : {len(submission)}")
    print(f"Saved to      : {output_path}")

    # Show sample output
    print("\nSample output (first 3 records):")
    print("-" * 50)
    for rec in submission[:3]:
        print(f"\n  sentence_id : {rec['sentence_id']}")
        print(f"  triplets    :")
        for t in rec["triplets"]:
            print(f"    Aspect={t['Aspect']}  Opinion={t['Opinion']}  "
                  f"Valence={t['Valence']}  Arousal={t['Arousal']}")
    print("-" * 50)

    return submission


# ── CHANGE PATHS BELOW AND RUN ───────────────────────────────────────
# Test file path (sir's file in your Drive):
TEST_FILE   = "/content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/test_data.jsonl"

# Output file path (where to save your submission):
OUTPUT_FILE = "/content/drive/MyDrive/DATASET/NLP/submission_output.jsonl"

# Run inference and generate submission file:
predictions = run_on_test_file(TEST_FILE, OUTPUT_FILE)


Reading test file : /content/drive/MyDrive/DATASET/NLP/EXAM DATA SET/test_data.jsonl
Total sentences   : 2000
Running inference...
  Processed 100/2000...
  Processed 200/2000...
  Processed 300/2000...
  Processed 400/2000...
  Processed 500/2000...
  Processed 600/2000...
  Processed 700/2000...
  Processed 800/2000...
  Processed 900/2000...
  Processed 1000/2000...
  Processed 1100/2000...
  Processed 1200/2000...
  Processed 1300/2000...
  Processed 1400/2000...
  Processed 1500/2000...
  Processed 1600/2000...
  Processed 1700/2000...
  Processed 1800/2000...
  Processed 1900/2000...
  Processed 2000/2000...

Done!
Total records : 2000
Saved to      : /content/drive/MyDrive/DATASET/NLP/submission_output.jsonl

Sample output (first 3 records):
--------------------------------------------------

  sentence_id : rest26_aste_test_119
  triplets    :
    Aspect=Service  Opinion=slow  Valence=3.27  Arousal=6.05
    Aspect=food  Opinion=slow  Valence=3.27  Arousal=6.05

  sentence_id : 